# 임베딩과 의미 유사도

단어가 다르더라도 뜻이 비슷한 문장을 숫자로 비교할 수 있을까요?

Python 3.12와 잠긴 패키지를 사용합니다. 첫 실행에서는 로컬 모델을 다운로드합니다.

## 다국어 문장 임베딩

문장을 384개 숫자로 바꾸는 로컬 모델을 불러옵니다.

In [1]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_ID, revision="e8f8c211226b894fcb81acc59f3b34ba3efd5f42")
sentences = [
    "주문한 상품을 취소하고 싶습니다.",
    "구매를 취소하려면 어떻게 하나요?",
    "내일 서울은 비가 옵니다.",
    "I want to cancel my order.",
]
vectors = model.encode(sentences, normalize_embeddings=True)
print("shape:", vectors.shape)
print("첫 문장 앞 5개 값:", np.round(vectors[0, :5], 4))

shape: (4, 384)
첫 문장 앞 5개 값: [ 0.0091  0.0092  0.0109 -0.0262 -0.0082]


## 코사인 유사도 비교

정규화된 벡터의 점곱은 코사인 유사도와 같습니다.

In [2]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
scores = vectors @ vectors.T
for i in range(1, len(sentences)):
    print(f"0 vs {i}: {scores[0, i]:.4f} | {sentences[i]}")
nearest = np.argsort(-scores[0])[1:]
print("\n유사도 순서:", nearest.tolist())

0 vs 1: 0.8024 | 구매를 취소하려면 어떻게 하나요?
0 vs 2: 0.1044 | 내일 서울은 비가 옵니다.
0 vs 3: 0.8987 | I want to cancel my order.

유사도 순서: [3, 1, 2]


## 단어 하나 바꾸기

'취소'를 '반품'으로 바꿔 유사도가 어떻게 달라지는지 봅니다.

In [3]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
changed = model.encode(["주문한 상품을 반품하고 싶습니다."], normalize_embeddings=True)[0]
print("취소 문장 vs 반품 문장:", round(float(vectors[0] @ changed), 4))
print("취소 문장 vs 날씨 문장:", round(float(vectors[0] @ vectors[2]), 4))

취소 문장 vs 반품 문장: 0.5931
취소 문장 vs 날씨 문장: 0.1044


## 구조 검증

예상한 실험 조건과 출력 구조가 유지되었는지 확인합니다.

In [4]:
assert vectors.shape == (4, 384) and np.isfinite(vectors).all()
assert scores[0, 3] > scores[0, 1] > scores[0, 2]
assert float(vectors[0] @ changed) > scores[0, 2]
print("검증: 벡터 구조와 본문의 유사도 순서를 확인했습니다.")

검증: 벡터 구조와 본문의 유사도 순서를 확인했습니다.
